# GPT-2 Text Generation - Fine Tuning
fine tuning gpt2 on custom data to generate text in similar style

In [1]:
# install required libs
!pip install transformers datasets accelerate -q

In [2]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from datasets import Dataset

print("gpu available:", torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'

gpu available: True


## Prepare Dataset
using some custom text, you can replace this with your own data

In [3]:
# my training text - can replace with anything
text = """
The sun sets slowly over the mountains, painting the sky in shades of orange and red.
Every evening brings a new canvas, a reminder that beauty exists in endings too.
The birds return to their nests, the wind grows still, and the world takes a breath.
In the silence that follows, one can hear the quiet rhythm of nature settling down.
Stars begin to appear, one by one, like old friends showing up to a familiar gathering.
"""

# repeating to get enough training data
full_text = text.strip() * 80

with open('my_data.txt', 'w') as f:
    f.write(full_text)

print('data ready, total chars:', len(full_text))

data ready, total chars: 33840


In [4]:
# load tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained('gpt2')
model = model.to(device)
print('model loaded, params:', sum(p.numel() for p in model.parameters()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

model loaded, params: 124439808


In [5]:
# tokenize the data
with open('my_data.txt', 'r') as f:
    raw = f.read()

tokens = tokenizer.encode(raw)
block = 128
chunks = [tokens[i:i+block] for i in range(0, len(tokens)-block, block)]

dataset = Dataset.from_dict({'input_ids': chunks, 'labels': chunks})
print('total chunks:', len(dataset))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7440 > 1024). Running this sequence through the model will result in indexing errors


total chunks: 58


## Training

In [6]:
args = TrainingArguments(
    output_dir='./output',
    num_train_epochs=4,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=50,
    save_steps=500,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=collator
)

trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,0.591372


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=60, training_loss=0.5013309667507807, metrics={'train_runtime': 16.6516, 'train_samples_per_second': 13.933, 'train_steps_per_second': 3.603, 'total_flos': 15154937856000.0, 'train_loss': 0.5013309667507807, 'epoch': 4.0})

In [7]:
# save the model
model.save_pretrained('./my_gpt2')
tokenizer.save_pretrained('./my_gpt2')
print('saved!')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved!


## Generate Text

In [8]:
# load and generate
gen_model = GPT2LMHeadModel.from_pretrained('./my_gpt2').to(device)
gen_tok = GPT2Tokenizer.from_pretrained('./my_gpt2')
gen_tok.pad_token = gen_tok.eos_token

def generate(prompt, max_tokens=100):
    inp = gen_tok(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = gen_model.generate(
            **inp,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.85,
            top_k=50,
            top_p=0.92,
            repetition_penalty=1.2,
            pad_token_id=gen_tok.eos_token_id
        )
    return gen_tok.decode(out[0], skip_special_tokens=True)

print(generate("The sun sets"))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The sun sets slowly over the mountains, painting sky in shades of orange and red.
Every evening brings a new canvas—a reminder that beauty exists ultimately too late for formula formulas to live up into power or comfort.*


In [9]:
# try different prompts
prompts = ["Stars begin", "Every evening", "The wind"]

for p in prompts:
    print(f'\nPrompt: {p}')
    print(generate(p, max_tokens=80))
    print('---')


Prompt: Stars begin
Stars begin to appear, one by a dying sun.
The world takes in view through the trees and into their homes: songs that echo throughout nature's beauty.Every evening brings an adventure for all of us—a reminder how powerful friendship can be when you keep up with them!
---

Prompt: Every evening
Every evening brings a new canvas, and one day will bring another.
In the silence that follows—a quiet rhythm grows between songs like old friends showing up to an in-temperance gathering.The sun sets slowly over mountains of orange color, painting everything from trees into shades reds and oranges until they become clear…
---

Prompt: The wind
The wind grows still, and the world takes a breath.
In the silence that follows, one can hear how quiet nature has become in response to our calls for attention.Stars begin their sets slowly over mountains or oceans; birds return home from nesting places like songbok. The sun set gradually through some of California's arbors, painting 

## Observations
- model picks up the writing style after fine tuning
- lower temperature = more predictable output
- more epochs = better style matching but risk of overfitting